# Part 2 – Curated Dataset

## Introduction

This project builds a curated dataset from historical application result data published by the Swedish National Agency for Higher Vocational Education (MYH).

The source data is spread across yearly Excel files. The files describe the same business process, but their sheets, header rows, column names, and available fields vary between years. Because of that, the data needs to be treated as a small data engineering project rather than a simple one-file pandas exercise.

The goal is to create a reusable curated dataset that can support downstream SQL storage, FastAPI endpoints, and dashboard analysis.

The work includes:

- schema inspection across years
- harmonization of columns and values
- data cleaning and text normalization
- enrichment of analytical fields
- validation and quality checks
- export to a curated dataset for SQL and API integration

The curated dataset is primarily based on Tabell 3. This table gives the most consistent application-level view, where each row represents one application record. That grain is suitable for a central applications table in PostgreSQL and for API filtering.


In [1]:
# Standard library
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


In [2]:
# Third-party libraries
import pandas as pd
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell

from src.myh_pipeline.config import (
    RAW_DATA_PATH,
    TARGET_COLUMNS,
    CURATED_DATA_PATH,
)

from src.myh_pipeline.load import (
    load_all_years,
    check_schema,
)

from src.myh_pipeline.clean import clean_all_years

from src.myh_pipeline.harmonize import harmonize_all_years

from src.myh_pipeline.validate import build_validation_summary

from src.myh_pipeline.enrich import enrich_dataset

In [3]:
# Jupyter display settings
InteractiveShell.ast_node_interactivity = "all"

## Step 1: Load and Combine Raw Excel Files

The project uses MYH application result files from 2021–2025.

These years were selected because they provide useful historical coverage while keeping the schema harmonization manageable. Earlier years can contain larger structural differences, but 2021–2025 still show variations in Excel layout and formatting while remaining suitable for building a unified multi-year dataset.

In this step, the raw Excel files are loaded from the raw data directory. A source year is extracted from the filename so each record can be traced back to its original file.

Although the selected years use mostly consistent column names, the Excel files still contain structural differences between years, such as varying header row positions and metadata rows before the actual tables.

Combining the yearly datasets makes it easier to:

- inspect structural differences between years
- identify layout inconsistencies
- define reusable import and cleaning logic
- standardize column naming
- prepare the data for one reusable curated table


In [4]:
# Excel source files
excel_files = RAW_DATA_PATH.glob("*.xlsx")

dfs = load_all_years()


Loading file: resultat-ansokningsomgang-2022.xlsx
Loading file: resultat-ansokningsomgang-2023.xlsx
Loading file: resultat-ansokningsomgang-2024.xlsx
Loading file: resultat-ansokningsomgang-2025.xlsx


## Step 2: Comparing Table Structures Across Years

The harmonization work focuses on Tabell 3 because it represents applications at a consistent business level.

Other sheets, such as Tabell 4, contain more detailed municipality-level relationships where the same application may appear more than once. Mixing those rows directly into the main table would change the grain and could create duplicate application records.

The inspection included:

- dataset dimensions
- column names
- header row differences
- structural differences between years

The comparison showed that the 2022 dataset contains fewer columns than the datasets from 2023–2025. This confirms that schema harmonization is needed before the data can become one curated dataset.


In [5]:
# Inspect each dataframe
schema_df = check_schema(dfs)
schema_df

,source_year,shape,columns
0,2022,"(1207, 19)","[Utbildningsområde, Utbildningsnamn, Beslut, D..."
1,2023,"(1258, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
2,2024,"(1272, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."
3,2025,"(1184, 31)","[Utbildningsområde, SUN5 inriktning, SUN5 inri..."


### Schema Comparison Summary

The historical Excel files share several core business concepts, but the structures are not fully consistent across years.

The main differences were:

- different header row positions across years
- slightly different column names for the same concept
- inconsistent naming conventions
- some variation in available columns

Because of this, a harmonized target schema was created and the source columns were mapped into that structure. This makes the later SQL table and API responses predictable.


## Step 3: Define Target Schema

Based on the schema comparison, a common target schema was defined for the curated dataset.

### Column Mapping Rationale

The source Excel files used slightly different column names across years for the same business concepts.

Examples:

| Original Columns | Harmonized Column |
|---|---|
| Utbildningsanordnare administrativ enhet | utbildningsanordnare |
| Studietakt % | studietakt_procent |
| Beslut | beslut |
| Studieform | studieform |

The mapping is based on meaning rather than exact text matching. This is important because the same field may be written differently from year to year.

The curated table keeps the original Swedish values, such as `beslut` and `studieform`, because they preserve source context and are useful when checking the data against the original files. It also adds normalized fields such as `decision_normalized` and `study_form_normalized` so SQL queries, API filters, and dashboard logic can use stable values.


In [6]:
print(f"Target column count: {len(TARGET_COLUMNS)}")

pd.DataFrame({"target_columns": TARGET_COLUMNS})

Target column count: 20


,target_columns
0,source_year
1,source_file
2,source_sheet
3,diarienummer
4,utbildningsnamn
5,utbildningsomrade
6,beslut
7,decision_normalized
8,kommun
9,lan


## Step 4: Column Standardization & Basic Cleaning

Column names were standardized before harmonization to reduce technical differences between years.

The cleaning rules included:

- converting all column names to lowercase
- removing leading and trailing spaces
- replacing Swedish characters with ASCII equivalents
- replacing spaces and special characters with underscores
- removing parentheses and percentage symbols where needed

Text normalization helps because SQL queries, API parameters, and Python transformations work better with predictable field names. It also reduces the risk of small formatting differences causing mapping or validation errors later in the pipeline.


In [7]:
# clean all years
standardized_dfs = clean_all_years(dfs)

# check result
schema_df = check_schema(standardized_dfs)
schema_df

,source_year,shape,columns
0,2022,"(1207, 19)","[utbildningsomrade, utbildningsnamn, beslut, d..."
1,2023,"(1258, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
2,2024,"(1272, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."
3,2025,"(1184, 31)","[utbildningsomrade, sun5_inriktning, sun5_inri..."


## Step 5: Schema Harmonization

After standardizing column names, the datasets still contained structural differences between years. In particular, the 2022 dataset included fewer columns than the datasets from 2023–2025.

To create a unified curated dataset, the schemas were harmonized into a common target structure.

The harmonization process included:

- aligning all datasets to the predefined target schema
- adding missing columns with null values
- keeping only relevant columns
- ensuring a consistent column order across all years
- adding source metadata columns for traceability

This step makes it possible to safely combine yearly data into one application-level table. The source metadata keeps the pipeline auditable because each row can still be connected back to its source year, file, and sheet.


In [8]:
# harmonize all dataframes
harmonized_dfs = harmonize_all_years(standardized_dfs)
schema_df = check_schema(harmonized_dfs)
schema_df


,source_year,shape,columns
0,2022,"(1207, 20)","[source_year, source_file, source_sheet, diari..."
1,2023,"(1258, 20)","[source_year, source_file, source_sheet, diari..."
2,2024,"(1272, 20)","[source_year, source_file, source_sheet, diari..."
3,2025,"(1184, 20)","[source_year, source_file, source_sheet, diari..."


## Step 6: Build the Curated Dataset

After harmonizing and cleaning the yearly datasets, the dataframes were combined into one unified curated dataset. The curated dataset was designed to function as the central source table for the SQL database and FastAPI service developed in Part 3.

At this stage, all datasets shared the same column structure, naming conventions, and normalized values. This made it possible to merge them into a single analysis-ready table without changing the application-level grain.

Additional enrichment was applied to improve the usability of the dataset for later analysis and API development. Examples include approval indicators, education length categories, sector categorization, and source-related metadata.

The final curated dataset represents harmonized application information across multiple years in a consistent and structured format. It can be validated, loaded into PostgreSQL, and served through the API.


In [9]:
curated_df = pd.concat(harmonized_dfs.values(), ignore_index=True)
curated_df = curated_df.convert_dtypes()
curated_df = enrich_dataset(curated_df)

print("-------- CURATED DATASET OVERVIEW --------")
print(f"Rows: {curated_df.shape[0]}")
print(f"Columns: {curated_df.shape[1]}")

curated_df.head()

-------- CURATED DATASET OVERVIEW --------
Rows: 4921
Columns: 24


,source_year,source_file,source_sheet,diarienummer,utbildningsnamn,utbildningsomrade,beslut,decision_normalized,kommun,lan,...,utbildningsanordnare,huvudmannatyp,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,is_approved,education_length,sector_category,record_source
0,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5458,.NET Cloud developer,Data/IT,Avslag,rejected,Stockholm,Stockholm,...,IT-Högskolan Stockholm AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
1,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4695,.NET Developer,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,...,KYH AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
2,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4476,.NET Utvecklare,Data/IT,Beviljad,approved,Gävle,Gävleborg,...,Plushögskolan AB - Teknikhögskolan,Privat,<NA>,<NA>,<NA>,<NA>,True,long,unknown,2022_Tabell 3
3,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4708,.NET Utvecklare,Data/IT,Avslag,rejected,Flera kommuner,Flera kommuner,...,Plushögskolan AB - Teknikhögskolan Mitt,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3
4,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/5529,.NET-utvecklare,Data/IT,Avslag,rejected,Kungälv,Västra Götaland,...,Optimum Education AB,Privat,<NA>,<NA>,<NA>,<NA>,False,long,unknown,2022_Tabell 3


### Aggregated Summary

Although most downstream statistics are calculated in SQL in Part 3, this notebook also includes a small pandas aggregation step to validate the curated dataset and demonstrate how the enriched fields can support analysis.

In [10]:
yearly_summary = (
    curated_df.groupby("source_year")
    .agg(
        total_applications=("diarienummer", "count"),
        approved_applications=("is_approved", "sum"),
        unique_providers=("utbildningsanordnare", "nunique"),
    )
    .reset_index()
)

yearly_summary["approval_rate"] = (
    yearly_summary["approved_applications"]
    / yearly_summary["total_applications"]
    * 100
).round(1)

yearly_summary

,source_year,total_applications,approved_applications,unique_providers,approval_rate
0,2022,1207,420,254,34.8
1,2023,1258,477,249,37.9
2,2024,1272,344,250,27.0
3,2025,1184,462,242,39.0


## Step 7: Validation and Quality Checks

The curated dataset was validated through both exploratory inspection and structured quality checks.

The validation process focused on:
- dataset completeness
- missing values
- duplicate detection
- application-level uniqueness
- required schema coverage
- categorical consistency in normalized fields

The duplicate application check uses the combination of source_year and diarienummer because this represents the practical business key for the curated applications dataset.

The required column check confirms that the curated dataset contains the core fields expected for the main applications table. Duplicate checks are performed at the application level using source year and diarienummer, because this combination represents the practical business key in the curated dataset.


In [11]:
print("--------- DATASET OVERVIEW ---------")
curated_df.info(show_counts=True)

print("\n--------- MISSING VALUES ---------")
display(curated_df.isna().sum())

print("\n--------- DUPLICATE ROWS ---------")
print(curated_df.duplicated().sum())

print("\n--------- PRIMARY KEY VALIDATION ---------")

missing_registry_numbers = curated_df["diarienummer"].isna().sum()
duplicate_registry_numbers = curated_df["diarienummer"].duplicated().sum()

print(f"Missing diarienummer values: {missing_registry_numbers}")
print(f"Duplicate diarienummer values: {duplicate_registry_numbers}")

# Validation confirms that diarienummer contains
# no missing or duplicate values and is therefore
# suitable as the relational primary key.

print("\n--------- REQUIRED COLUMN CHECK ---------")

required_columns = [
    "source_year",
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "decision_normalized",
    "kommun",
    "lan",
    "yh_poang",
    "studieform",
    "studietakt_procent",
    "utbildningsanordnare",
    "huvudmannatyp",
]

missing_required_columns = [
    col for col in required_columns
    if col not in curated_df.columns
]

print(missing_required_columns)

print("\n--------- LOW-CARDINALITY COLUMN INSPECTION ---------")

inspect_columns = [
    "decision_normalized",
    "study_form_normalized",
    "huvudmannatyp",
    "sector_category",
]

for col in inspect_columns:
    print(f"\n--- {col} ---")

    value_counts_df = curated_df[col].value_counts(
        dropna=False
    ).reset_index()

    value_counts_df.columns = [col, "count"]

    display(value_counts_df)


--------- DATASET OVERVIEW ---------
<class 'pandas.DataFrame'>
RangeIndex: 4921 entries, 0 to 4920
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype        
---  ------                 --------------  -----        
 0   source_year            4921 non-null   Int64        
 1   source_file            4921 non-null   string       
 2   source_sheet           4921 non-null   string       
 3   diarienummer           4921 non-null   string       
 4   utbildningsnamn        4921 non-null   string       
 5   utbildningsomrade      4921 non-null   string       
 6   beslut                 4921 non-null   string       
 7   decision_normalized    4921 non-null   string       
 8   kommun                 4921 non-null   string       
 9   lan                    4921 non-null   string       
 10  yh_poang               4921 non-null   Int64        
 11  studieform             4921 non-null   string       
 12  study_form_normalized  4921 non-null   string     

source_year                 0
source_file                 0
source_sheet                0
diarienummer                0
utbildningsnamn             0
utbildningsomrade           0
beslut                      0
decision_normalized         0
kommun                      0
lan                         0
yh_poang                    0
studieform                  0
study_form_normalized       0
studietakt_procent          0
utbildningsanordnare        0
huvudmannatyp               0
sun5_inriktning          1207
sun5_inriktning_namn     1207
seqf_niva                1260
smalt_yrkesomrade        1207
is_approved                 0
education_length            0
sector_category             0
record_source               0
dtype: int64


--------- DUPLICATE ROWS ---------
0

--------- PRIMARY KEY VALIDATION ---------
Missing diarienummer values: 0
Duplicate diarienummer values: 0

--------- REQUIRED COLUMN CHECK ---------
[]

--------- LOW-CARDINALITY COLUMN INSPECTION ---------

--- decision_normalized ---


,decision_normalized,count
0,rejected,3217
1,approved,1703
2,withdrawn,1



--- study_form_normalized ---


,study_form_normalized,count
0,on_site,2572
1,distance,2349



--- huvudmannatyp ---


,huvudmannatyp,count
0,Privat,4073
1,Kommun,779
2,Region,60
3,Statlig,9



--- sector_category ---


,sector_category,count
0,unknown,1207
1,data_it,988
2,other,537
3,ekonomi_forsaljning,523
4,bygg_fastighet_vvs,472
5,teknik_industri,443
6,halso_sjukvard,308
7,pedagogik_socialt_arbete,173
8,transport,98
9,media_design_kultur,90


## Formal Validation Summary

In addition to exploratory inspection, a structured validation summary was created to identify potential data quality issues in a standardized and reusable way.

The checks also include text length validation against the planned PostgreSQL schema constraints. This helps catch values that could fail during database loading or cause inconsistent API responses later.


In [12]:
validation_summary = build_validation_summary(curated_df)

display(validation_summary)

,check,affected_rows,severity,max_length_found
0,missing_diarienummer,0,critical,NaN
1,duplicate_diarienummer,0,warning,NaN
2,missing_utbildningsanordnare,0,warning,NaN
3,invalid_decision_values,0,warning,NaN
4,invalid_study_form_values,0,warning,NaN
5,missing_sun5_fields,1207,info,NaN
6,utbildningsnamn_max_length,0,warning,106.0
7,utbildningsanordnare_max_length,0,warning,73.0
8,sun5_inriktning_namn_max_length,0,warning,87.0


## Step 8: Export Curated Dataset

After validation and quality checks, the curated dataset was exported for further use.

The exported CSV represents the cleaned, harmonized, enriched, and combined dataset. It is the handoff point between the pandas pipeline and the later SQL/FastAPI part of the project.

Exporting the curated data makes the transformation reproducible: the same source files and pipeline rules can be rerun to produce a fresh dataset when the raw files change or when new years are added.


In [13]:
CURATED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

curated_df.to_csv(
    CURATED_DATA_PATH / "curated_applications.csv",
    index=False,
)

## Reflection

One of the main challenges in this project was balancing historical consistency with practical usability. The yearly MYH files describe similar business concepts, but the structures and naming conventions differ enough that explicit harmonization rules are needed.

A key design choice was to keep the curated table at application-level grain. This makes the dataset easier to load into PostgreSQL, expose through FastAPI, and analyze in a dashboard without accidentally duplicating applications.

Another useful pattern was keeping both original Swedish source values and normalized analytical fields. The original values support traceability and review, while the normalized fields make filtering, aggregation, and API usage more reliable.

Overall, the result is more than a cleaned CSV. It is a reproducible transformation pipeline that creates a reusable curated dataset for downstream database, API, and analytics work.
